In [ ]:
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt

# Histogram equalization demo

In [ ]:
!wget -nc https://www.dais.unive.it/~bergamasco/teachingfiles/cvstuff/1.jpg

In [ ]:
I = cv.imread("1.jpg", cv.IMREAD_GRAYSCALE)
print( type(I))
print( I.shape )

plt.figure( figsize=(10,10))
plt.imshow( I, cmap="gray", vmin=0, vmax=255 )

In [ ]:
# manually compute the image histogram
H = np.zeros( 256, dtype=np.uint32 )

for y in range( I.shape[0]):
    for x in range(I.shape[1]):
        H[ I[y,x] ] += 1

print("Image histogram: ")
print(H)

plt.figure( figsize=(10,5))
plt.plot( H )
plt.xlabel("Intensity level")
plt.ylabel("Number of pixels")
plt.grid(True)

In [ ]:
# Compute the equalization function T

T = np.zeros( 256, dtype=np.uint32 )

for ii in range( 256 ):
    T[ii] = np.sum( H[:ii+1] ) * 255 / ( I.shape[0]*I.shape[1])

print("T(r) = ")
print(T)

In [ ]:
# Apply T to the original image

I_eq = np.zeros_like( I )

for y in range( I.shape[0]):
    for x in range(I.shape[1]):
        I_eq[y,x] = T[ I[y,x] ]

plt.figure( figsize=(10,10))
plt.imshow( I_eq, cmap="gray", vmin=0, vmax=255 )


H_eq = np.zeros( 256, dtype=np.uint32 )
for p in I_eq:
    H_eq[p] += 1


plt.figure( figsize=(10,5))
plt.plot( H_eq )
plt.xlabel("Intensity level")
plt.ylabel("Number of pixels")
plt.grid(True)

# Histogram matching demo

In [ ]:
!wget -nc https://www.dais.unive.it/~bergamasco/teachingfiles/cvstuff/2.jpg

In [ ]:
I1 = cv.imread( "1.jpg", cv.IMREAD_GRAYSCALE)
I2 = cv.imread( "2.jpg", cv.IMREAD_GRAYSCALE)

Compute normalized [histograms](https://docs.opencv.org/4.x/d6/dc7/group__imgproc__hist.html#ga6ca1876785483836f72a77ced8ea759a)

In [ ]:
hist1 = cv.calcHist([I1],[0],None,[256],[0,256])/np.prod(I1.shape)
hist2 = cv.calcHist([I2],[0],None,[256],[0,256])/np.prod(I2.shape)

In [ ]:
plt.figure( figsize=(20,10))
plt.subplot(2,2,1)
plt.imshow(I1, cmap="gray", vmin=0, vmax=255)
plt.title("I1")
plt.subplot(2,2,2)
plt.title("I1 normalized histogram")
plt.plot(hist1)
plt.grid(True)
plt.subplot(2,2,3)
plt.imshow(I2, cmap="gray",vmin=0, vmax=255)
plt.title("I2")
plt.subplot(2,2,4)
plt.title("I2 normalized histogram")
plt.plot(hist2)
plt.grid(True)

Compute the CDF

In [ ]:
cdf1 = np.cumsum(hist1)
cdf2 = np.cumsum(hist2)

In [ ]:
plt.figure( figsize=(20,10))
plt.subplot(2,2,1)
plt.imshow(I1, cmap="gray", vmin=0, vmax=255)
plt.title("I1")
plt.subplot(2,2,2)
plt.title("I1 CDF")
plt.plot(cdf1)
plt.grid(True)
plt.subplot(2,2,3)
plt.imshow(I2, cmap="gray",vmin=0, vmax=255)
plt.title("I2")
plt.subplot(2,2,4)
plt.title("I2 CDF")
plt.plot(cdf2)
plt.grid(True)

Equalize image 1

In [ ]:
CDF1_LUT = (cdf1*255).astype(np.uint8)
I1_eq = CDF1_LUT[I1]

plt.figure()
plt.subplot(1,2,1)
plt.imshow(I1, cmap="gray", vmin=0, vmax=255)
plt.subplot(1,2,2)
plt.imshow(I1_eq, cmap="gray", vmin=0, vmax=255)

Equalize image 2

In [ ]:
CDF2_LUT = (cdf2*255).astype(np.uint8)
I2_eq = CDF2_LUT[I2]

plt.figure()
plt.subplot(1,2,1)
plt.imshow(I2, cmap="gray", vmin=0, vmax=255)
plt.subplot(1,2,2)
plt.imshow(I2_eq, cmap="gray", vmin=0, vmax=255)

Compute ```G```, as the function mapping I2 to I2_equalized

In [ ]:
G_in = np.arange(256)
G_out = CDF2_LUT
#print( np.concatenate( [np.expand_dims(G_in, axis=1), np.expand_dims(G_out, axis=1)], axis=1))
plt.figure()
plt.plot( G_in, G_out)
plt.grid(True)
plt.xlabel("G_in")
plt.ylabel("G_out")

Invert G to map from equalized I2 to I2
note: the inversion may not be exact since G is not always strictly monotonic

In [ ]:
uniq_vals, uniq_indices,uniq_inverse = np.unique(G_out, return_index=True, return_inverse=True ) # <- Here is where we let the function be monotonic

Ginv_in = G_out[uniq_indices]
Ginv_out = G_in[uniq_indices]

# Now G is inverted but can contain missing values
#print( np.concatenate( [np.expand_dims(Ginv_in, axis=1), np.expand_dims(Ginv_out, axis=1)], axis=1))
plt.figure()
plt.plot( Ginv_in, Ginv_out, '.')
plt.grid(True)
plt.ylabel("G_in")
plt.xlabel("G_out")

Interpolate missing values

In [ ]:
import scipy.interpolate

interpolator = scipy.interpolate.interp1d( Ginv_in, Ginv_out, kind="linear")

Ginv_in = np.arange(256)
Ginv_out = interpolator( Ginv_in )

plt.figure()
plt.plot( Ginv_in, Ginv_out,'.')
plt.grid(True)
plt.ylabel("G_in")
plt.xlabel("G_out")

Apply Ginv to the equalized I1

In [ ]:
I1_to_I2 = Ginv_out[I1_eq].astype(np.uint8)
I1_to_I2_hist = cv.calcHist([I1_to_I2],[0],None,[256],[0,256])/np.prod(I1_to_I2.shape)

plt.figure( figsize=(20,10) )
plt.subplot(2,3,1)
plt.imshow(I1, cmap="gray", vmin=0, vmax=255)
plt.title("I1 (original)")
plt.subplot(2,3,2)
plt.imshow(I1_to_I2, cmap="gray", vmin=0, vmax=255)
plt.title("I1 matched to I2")
plt.subplot(2,3,3)
plt.imshow(I2, cmap="gray", vmin=0, vmax=255)
plt.title("I2 (original)")
plt.subplot(2,3,4)
plt.fill_between(np.arange(256),np.squeeze(hist1))
plt.grid(True)
plt.subplot(2,3,5)
plt.fill_between(np.arange(256),np.squeeze(I1_to_I2_hist))
plt.grid(True)
plt.subplot(2,3,6)
plt.fill_between(np.arange(256),np.squeeze(hist2))
plt.grid(True)

In [ ]:
!rm 1.jpg 2.jpg